# Concentration GUI example

Click the wells in a **popup window**, then this notebook analyzes them.
Plots and images stay **inside the notebook**.

**How to run**

1. Run the cells from top to bottom.
2. In *Choose your image*, either keep the example image or uncomment `IMAGE_PATH`.
3. When the well-selection window opens, click the wells, then press **Enter**.
4. The remaining cells show plots here. If saving is on, files go in a results folder.


## 1. Setup

This cell loads QAL and keeps later plots inside the notebook.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qal import WellAnalyzer, WellPlotter, select_concentration_grid
from qal.data import cn_sample_1, get_concentration_target_preset
from qal.util import load_grayscale_image

from qal.util import save_sample_sources, save_target_metadata


## 2. Choose your image

`IMAGE_PATH = cn_sample_1()` loads the built-in example image.

To use your own image, comment out that line and set `IMAGE_PATH` to a
TIFF, BMP, PNG, or JPEG file path. Leave only one option active.
The relative file path works from this notebook's folder in a GitHub checkout or download.

`None` raises an error because no image has been supplied.

**Quantitative image input:** this example passes the input through
`load_grayscale_image` before analysis. Native 2-D grayscale values and dtype
are preserved without rescaling. RGB/RGBA inputs use scikit-image rgb2gray luminance weights; alpha is composited onto white before grayscale conversion.


In [ ]:
# Images are loaded and converted to grayscale with load_grayscale_image.
# Already-grayscale images remain unchanged: stored values and dtype are preserved.
# RGB/RGBA inputs use scikit-image rgb2gray luminance weights; alpha is composited onto white before grayscale conversion.
# Choose one option; leave the others commented out.
# Uncomment the next line and put the path to your image:
# IMAGE_PATH = "/path/to/your/image.tiff"

# Example using a file that ships with QAL:
# Run from this example's directory in a GitHub checkout or download.
# IMAGE_PATH = "../../data/concentration_targets/cn_sample_1.tiff"

# Selecting the built-in example image:
IMAGE_PATH = cn_sample_1()


## 3. Saving plots and data

These are the knobs that control whether files are written, and what they are
called. When `SAVE_RESULTS` is `True`:

- If you set `IMAGE_PATH` to a file path, files go next to that image in `<image_name>_results/`
- If you use the built-in image, files go next to this notebook in `cn_sample_1_results/`

Set `SAVE_SOURCE_IMAGES = True` to also copy the original built-in sample
files. User-supplied images are not copied. Multiple source images are grouped
in `source_images/`. This does not move or change cached data.


In [ ]:
# Set this to True to write plots and CSV files.
SAVE_RESULTS = True
# Optional: copy original built-in samples; user-supplied images are not copied.
SAVE_SOURCE_IMAGES = False

# File names inside the results folder (used only when SAVE_RESULTS is True):
SAVE_ROI_PLOT = "roi.png"
SAVE_CONCENTRATION_PLOT = "concentration.png"
SAVE_STATS_CSV = "stats.csv"


## 4. Analysis options

You can usually leave these as they are.

Click order for the first three wells (left to right along the top row):

`1000 nM` → `300 nM` → `100 nM`


In [ ]:
# Product type: "icg", "q800", or "q700"
TARGET_TAG = "icg"

# How wells are selected. Uncomment one:
# MODE = "automatic_refinement"
MODE = "manual_live_preview"
# MODE = "manual_coordinates"

# How many wells to click (3 is enough; they must be left-to-right, top-to-bottom).
ANCHOR_COUNT = 3

# Circle size in pixels. None = estimate from well spacing.
MANUAL_ROI_DIAMETER = None

# Fraction of each well used for intensity (0.5 is recommended).
REGION_OF_WELL_TO_ANALYZE = 0.5


## 5. Load the image


In [ ]:
if IMAGE_PATH is None:
    raise ValueError("No image has been supplied. Set IMAGE_PATH to an image file path or a built-in example call.")

if isinstance(IMAGE_PATH, np.ndarray):
    image = load_grayscale_image(IMAGE_PATH)
    results_dir = Path("cn_sample_1_results")
else:
    image = load_grayscale_image(IMAGE_PATH)
    image_file = Path(IMAGE_PATH).expanduser()
    results_dir = image_file.parent / f"{image_file.stem}_results"

if SAVE_RESULTS:
    results_dir.mkdir(parents=True, exist_ok=True)
    print(f"Results folder: {results_dir.resolve()}")
else:
    results_dir = None

target = get_concentration_target_preset(TARGET_TAG)
well_ids = target["well_ids"]
print(f"Target: {target['display_name']}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(image, cmap="gray")
ax.set_title("Loaded image")
ax.axis("off")
plt.show()


## 6. Select wells (popup window)

This cell opens a window. Click the wells, then press **Enter**. Drag a `+` to move it.

Later plots stay in this notebook.


In [ ]:
wells = select_concentration_grid(
    image,
    well_ids,
    mode=MODE,
    anchor_count=ANCHOR_COUNT,
    manual_roi_diameter=MANUAL_ROI_DIAMETER,
)


## 7. Analyze the selected wells


In [ ]:
if not isinstance(wells, pd.DataFrame):
    raise RuntimeError(
        "No well table yet. Finish the popup with Enter, then rerun "
        "the select-wells cell."
    )

analyzer = WellAnalyzer(image, wells)
stats = analyzer.get_stats(region_of_well_to_analyze=REGION_OF_WELL_TO_ANALYZE)
stats


## 8. Show the regions of interest


In [ ]:
plotter = WellPlotter(stats, image=image)
roi_path = results_dir / SAVE_ROI_PLOT if results_dir is not None else None
plotter.visualize_roi(save_plot=roi_path)
if results_dir is not None:
    print(f"Saved {roi_path}")


## 9. Plot concentration vs intensity


In [ ]:
plotter.plot(
    graph_type="concentration",
    col_to_plot="mean intensity normalized",
    fluorophore_label=target.get("fluorophore_label", "ICG-eq"),
    trendline_lib="statsmodels",
    save_plot=(
        None if results_dir is None else results_dir / SAVE_CONCENTRATION_PLOT
    ),
)


## 10. Save the analysis table as CSV

This writes the post-analysis metrics (including well coordinates) into the
results folder, the same way `concentration_gui_example.py` does.


In [ ]:
if results_dir is None:
    print("SAVE_RESULTS is False, so no CSV files were written.")
else:
    stats_path = results_dir / SAVE_STATS_CSV
    stats.to_csv(stats_path, index=False)
    save_target_metadata(results_dir, target)
    print(f"Saved {stats_path}")

    if SAVE_SOURCE_IMAGES:
        save_sample_sources(results_dir, IMAGE_PATH)
